# 02. Thực nghiệm Tiền xử lý Dữ liệu Tiếng Việt Đặc thù (Vietnamese NLP Preprocessing)
Notebook này trình bày chi tiết quy trình làm sạch, chuẩn hóa và tách từ tiếng Việt trong `src/preprocess.py`:
1. **Xử lý Missing Values**: Điền chuỗi rỗng `""` cho 3 mẫu thiếu `lead_paragraph` (`article_2299`, `article_1821`, `article_2870`).
2. **Khử lỗi dính địa danh ở đầu `lead_paragraph` của VnExpress** (`165 mẫu` bị dính tên tỉnh/thành phố/quốc gia vào từ đầu tiên của câu, ví dụ: `"Nghệ AnCảng hàng không..."` -> `"Nghệ An Cảng hàng không..."`).
3. **Khử lỗi dính hậu tố tên nguồn báo ở cuối `title` của Báo Mới** (`716 mẫu` có đuôi `" - Báo ..."`, `" - Tạp chí ..."`).
4. **Chuẩn hóa Unicode dựng sẵn (`NFC`), thống nhất vị trí dấu thanh và tách từ tiếng Việt bằng `PyVi` (`ViTokenizer.tokenize`)**.
5. **Thiết lập 2 phương án đầu vào**: `Option A (Only Title)` và `Option B (Title + [SEP] + Lead Paragraph)`, chia tập dữ liệu `70% Train - 15% Validation - 15% Test` bằng `StratifiedShuffleSplit (random_state=42)`.

In [ ]:
import os
import sys
import json
import pandas as pd

sys.path.append(os.path.abspath('..'))
from src.preprocess import (
    fix_vnexpress_glued_location,
    remove_baomoi_source_suffix,
    clean_title,
    clean_lead_paragraph,
    segment_vietnamese,
)

with open('../outputs/results/preprocessing_summary.json', 'r', encoding='utf-8') as f:
    prep_summary = json.load(f)

print(f"Tổng số mẫu xử lý: {prep_summary['total_samples']}")
print(f"Số mẫu sửa lỗi dính địa danh VnExpress: {prep_summary['vnexpress_glued_fixed_count']}")
print(f"Số mẫu cắt đuôi tên báo Báo Mới: {prep_summary['baomoi_suffix_fixed_count']}")

## 1. Minh chứng 5 câu TRƯỚC và SAU khi xử lý lỗi dính địa danh (VnExpress)

In [ ]:
pd.set_option('display.max_colwidth', None)
vne_df = pd.DataFrame(prep_summary['vnexpress_examples'])
display(vne_df)

## 2. Minh chứng 5 câu TRƯỚC và SAU khi xử lý lỗi đuôi tên báo (Báo Mới)

In [ ]:
bm_df = pd.DataFrame(prep_summary['baomoi_examples'])
display(bm_df)

## 3. Minh chứng Tách từ tiếng Việt (Word Segmentation với `PyVi`) và Tạo Input Option A & Option B

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv', encoding='utf-8')
val_df = pd.read_csv('../data/processed/val.csv', encoding='utf-8')
test_df = pd.read_csv('../data/processed/test.csv', encoding='utf-8')

display(train_df[['id', 'title_clean', 'title_seg', 'input_option_b', 'label']].head(5))
print('=== Thống kê phân chia tập dữ liệu (Stratified 70/15/15) ===')
display(pd.DataFrame(prep_summary['splits']).T)